<a href="https://colab.research.google.com/github/ketanasindhusomisetty/Machine_Learning_2520030487/blob/main/Skill/Clustering_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
"""
Iris — Flower Clustering: K-Means, Hierarchical, DBSCAN
=====================================================

Goal: cluster iris flowers into botanical "profiles" WITHOUT using the
Species label -- i.e. unsupervised segmentation. Species is used only
*afterwards* to describe each cluster (class distribution per profile),
never as a clustering feature.

>>> IMPORTANT: edit the filename in the pd.read_csv(...) call below (in
>>> section 1) so it exactly matches your CSV file's name and location.

Outputs (written to the current working directory):
    - kmeans_elbow_silhouette.png
    - kmeans_pca.png
    - hierarchical_dbscan_pca.png
    - dendrogram.png
    - cluster_profiles.csv
    - iris_with_clusters.csv
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # comment this out if running interactively and you want inline plots
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FEATURE_COLS = [
    "SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"
]
TARGET_COL = "Species"

print("=" * 70)
print("IRIS FLOWER CLUSTERING: K-Means / Hierarchical / DBSCAN")
print("=" * 70)

IRIS FLOWER CLUSTERING: K-Means / Hierarchical / DBSCAN


In [7]:
# ---------------------------------------------------------------------------
# 1. Load + preprocess
# ---------------------------------------------------------------------------
# >>> EDIT the filename below so it exactly matches your CSV file <<<
df = pd.read_csv("/content/Iris (2).csv")

if "Id" in df.columns:
    df = df.drop(columns=["Id"])

imputer = SimpleImputer(strategy="median")
feature_df = pd.DataFrame(
    imputer.fit_transform(df[FEATURE_COLS]), columns=FEATURE_COLS, index=df.index
)

X = StandardScaler().fit_transform(feature_df)

print(f"Loaded {len(df)} flowers, {feature_df.shape[1]} clustering features "
      f"({TARGET_COL} excluded)")

Loaded 150 flowers, 4 clustering features (Species excluded)


In [8]:
# ---------------------------------------------------------------------------
# 2. Choose k for K-Means via elbow + silhouette
# ---------------------------------------------------------------------------
k_range = range(2, 6)
inertias, sil_scores = [], []
for k in k_range:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = km_k.fit_predict(X)
    inertias.append(km_k.inertia_)
    sil_scores.append(silhouette_score(X, labels_k))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(list(k_range), inertias, marker="o")
ax1.set_title("Elbow method (inertia)")
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")

ax2.plot(list(k_range), sil_scores, marker="o", color="darkorange")
ax2.set_title("Silhouette score vs k")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette score")

plt.tight_layout()
plt.savefig("kmeans_elbow_silhouette.png", dpi=150)
print("Saved plot -> kmeans_elbow_silhouette.png")
plt.close(fig)

best_k = list(k_range)[int(np.argmax(sil_scores))]
print(f"Silhouette-suggested k = {best_k}  (scores: "
      + ", ".join(f"k={k}:{s:.3f}" for k, s in zip(k_range, sil_scores)) + ")")

Saved plot -> kmeans_elbow_silhouette.png
Silhouette-suggested k = 2  (scores: k=2:0.580, k=3:0.459, k=4:0.385, k=5:0.347)


In [9]:
# ---------------------------------------------------------------------------
# 3. K-Means on the dataset
# ---------------------------------------------------------------------------
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
km_labels = kmeans.fit_predict(X)

sil = silhouette_score(X, km_labels)
print(f"\nK-Means (k={best_k}) silhouette: {sil:.3f}")
print("Cluster sizes:", pd.Series(km_labels).value_counts().sort_index().to_dict())

# --- Profile clusters against Species (label used only for reporting) ---
profiled = feature_df.copy()
profiled["KMeansCluster"] = km_labels
profiled["Species"] = df["Species"].values

summary = profiled.groupby("KMeansCluster").agg(
    n_flowers=("Species", "size"),
    avg_SepalLengthCm=("SepalLengthCm", "mean"),
    avg_SepalWidthCm=("SepalWidthCm", "mean"),
    avg_PetalLengthCm=("PetalLengthCm", "mean"),
    avg_PetalWidthCm=("PetalWidthCm", "mean"),
).round(3)

print("\n--- Cluster profiles (KMeansCluster) ---")
print(summary.to_string())
summary.to_csv("cluster_profiles.csv")
print("Saved table -> cluster_profiles.csv")


K-Means (k=2) silhouette: 0.580
Cluster sizes: {0: 100, 1: 50}

--- Cluster profiles (KMeansCluster) ---
               n_flowers  avg_SepalLengthCm  avg_SepalWidthCm  avg_PetalLengthCm  avg_PetalWidthCm
KMeansCluster                                                                                     
0                    100              6.262             2.872              4.906             1.676
1                     50              5.006             3.418              1.464             0.244
Saved table -> cluster_profiles.csv


In [10]:
# ---------------------------------------------------------------------------
# 4. Hierarchical + DBSCAN
# ---------------------------------------------------------------------------
hc = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hc_labels = hc.fit_predict(X)
sil_hc = silhouette_score(X, hc_labels)
print(f"\nHierarchical (k={best_k}) silhouette: {sil_hc:.3f}")

db = DBSCAN(eps=0.7, min_samples=5)
db_labels = db.fit_predict(X)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int(np.sum(db_labels == -1))
print(f"DBSCAN: {n_db_clusters} clusters, {n_noise} noise points")
if n_db_clusters >= 2:
    mask = db_labels != -1
    print(f"DBSCAN silhouette (excl. noise): {silhouette_score(X[mask], db_labels[mask]):.3f}")

# --- Dendrogram ---
Z = linkage(X, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z)
plt.title("Hierarchical Clustering Dendrogram (Ward linkage)")
plt.xlabel("Flower index")
plt.ylabel("Distance")
plt.tight_layout()
plt.savefig("dendrogram.png", dpi=150)
print("Saved plot -> dendrogram.png")
plt.close()


Hierarchical (k=2) silhouette: 0.575
DBSCAN: 2 clusters, 8 noise points
DBSCAN silhouette (excl. noise): 0.610
Saved plot -> dendrogram.png


In [11]:
# ---------------------------------------------------------------------------
# 5. PCA for visualization
# ---------------------------------------------------------------------------
X_pca_full = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)

# K-Means plot
fig, ax = plt.subplots(figsize=(6, 5.2))
ax.scatter(X_pca_full[:, 0], X_pca_full[:, 1], c=km_labels, cmap="tab10", s=30, edgecolor="k", alpha=0.8)
ax.set_title(f"K-Means, full data ({best_k} clusters)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig("kmeans_pca.png", dpi=150)
print("Saved plot -> kmeans_pca.png")
plt.close(fig)

# Hierarchical + DBSCAN plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))

axes[0].scatter(X_pca_full[:, 0], X_pca_full[:, 1], c=hc_labels, cmap="tab10", s=30, edgecolor="k", alpha=0.8)
axes[0].set_title(f"Hierarchical ({best_k} clusters)")

noise_mask = db_labels == -1
axes[1].scatter(
    X_pca_full[~noise_mask, 0], X_pca_full[~noise_mask, 1],
    c=db_labels[~noise_mask], cmap="tab10", s=30, edgecolor="k", alpha=0.8,
)
axes[1].scatter(
    X_pca_full[noise_mask, 0], X_pca_full[noise_mask, 1],
    c="lightgray", s=30, edgecolor="gray", marker="x", label="noise",
)
axes[1].legend(loc="upper right", fontsize=8)
axes[1].set_title(f"DBSCAN ({n_db_clusters} clusters)")

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.savefig("hierarchical_dbscan_pca.png", dpi=150)
print("Saved plot -> hierarchical_hierarchical_dbscan_pca.png")
plt.close(fig)

Saved plot -> kmeans_pca.png


/tmp/ipykernel_2857/372060341.py:28: UserWarning: You passed a edgecolor/edgecolors ('gray') for an unfilled marker ('x').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  axes[1].scatter(


Saved plot -> hierarchical_hierarchical_dbscan_pca.png


In [12]:
# ---------------------------------------------------------------------------
# 6. Save deliverable: original data + cluster assignment
# ---------------------------------------------------------------------------
df_out = df.copy()
df_out["KMeansCluster"] = km_labels
df_out.to_csv("iris_with_clusters.csv", index=False)
print("\nSaved deliverable -> iris_with_clusters.csv "
      "(original data + KMeansCluster column)")

print("\nDone. K-Means gives a clean segmentation of iris flowers into profiles; "
      "the species distribution per cluster (see cluster_profiles.csv) "
      "shows how botanical classes map to unsupervised clusters.")


Saved deliverable -> iris_with_clusters.csv (original data + KMeansCluster column)

Done. K-Means gives a clean segmentation of iris flowers into profiles; the species distribution per cluster (see cluster_profiles.csv) shows how botanical classes map to unsupervised clusters.
